In [1]:
#train mental health model

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib

# Load dataset
df = pd.read_csv(r"D:\capstone project\data\mental_health_dataset.csv")

# Encode gender + label
le_gender = LabelEncoder()
df["gender"] = le_gender.fit_transform(df["gender"])

le_state = LabelEncoder()
df["mental_state"] = le_state.fit_transform(df["mental_state"])

# Features + target
X = df.drop("mental_state", axis=1)
y = df["mental_state"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# Save everything
joblib.dump(model, "mental_health_model.pkl")
joblib.dump(le_gender, "gender_encoder.pkl")
joblib.dump(le_state, "state_encoder.pkl")

print("Model + encoders saved.")


              precision    recall  f1-score   support

           0       0.96      0.72      0.82        61
           1       0.73      0.91      0.81        85
           2       0.79      0.76      0.78        50
           3       0.00      0.00      0.00         4

    accuracy                           0.80       200
   macro avg       0.62      0.60      0.60       200
weighted avg       0.80      0.80      0.79       200

Model + encoders saved.


d:\capstone project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\capstone project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\capstone project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [2]:
#journal text to feature extraction

import nltk
import re
from nltk.corpus import stopwords
from textblob import TextBlob

nltk.download("stopwords")
STOPWORDS = set(stopwords.words("english"))

# Keyword dictionaries
STRESS_WORDS = ["tired", "pressure", "deadline", "overwhelmed", "burnout"]
ANXIETY_WORDS = ["nervous", "worried", "panic", "fear", "anxious"]
DEPRESS_WORDS = ["sad", "hopeless", "empty", "cry", "lonely"]
SLEEP_WORDS = ["sleep", "insomnia", "rest", "dream"]
SOCIAL_WORDS = ["friend", "family", "talk", "support"]
ACTIVITY_WORDS = ["walk", "gym", "run", "exercise"]
WORK_WORDS = ["office", "boss", "job", "project", "meeting"]

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

def keyword_score(text, keywords):
    return sum(text.count(word) for word in keywords)

def extract_features_from_journal(journal_text):
    text = clean_text(journal_text)

    sentiment = TextBlob(text).sentiment.polarity  # -1 to 1

    stress = keyword_score(text, STRESS_WORDS) + max(0, -sentiment * 5)
    anxiety = keyword_score(text, ANXIETY_WORDS) + max(0, -sentiment * 4)
    depression = keyword_score(text, DEPRESS_WORDS) + max(0, -sentiment * 6)
    sleep_quality = 10 - keyword_score(text, SLEEP_WORDS)
    social_support = keyword_score(text, SOCIAL_WORDS)
    physical_activity = keyword_score(text, ACTIVITY_WORDS)
    work_pressure = keyword_score(text, WORK_WORDS)

    # Clamp 0–10
    features = {
        "stress": min(10, int(stress)),
        "anxiety": min(10, int(anxiety)),
        "depression": min(10, int(depression)),
        "sleep_quality": max(0, min(10, int(sleep_quality))),
        "social_support": min(10, int(social_support)),
        "physical_activity": min(10, int(physical_activity)),
        "work_pressure": min(10, int(work_pressure))
    }

    return features


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aksha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [3]:
#prediction pipeline

import joblib
import pandas as pd

# Load saved model + encoders
model = joblib.load("mental_health_model.pkl")
le_gender = joblib.load("gender_encoder.pkl")
le_state = joblib.load("state_encoder.pkl")

def predict_from_journal(age, gender, journal_text):
    features = extract_features_from_journal(journal_text)

    gender_encoded = le_gender.transform([gender])[0]

    input_data = {
        "age": age,
        "gender": gender_encoded,
        **features
    }

    df_input = pd.DataFrame([input_data])
    pred = model.predict(df_input)[0]
    proba = model.predict_proba(df_input).max()

    mental_state = le_state.inverse_transform([pred])[0]

    score = round(proba * 100, 2)

    return {
        "mental_state": mental_state,
        "confidence_score": score,
        "features": features
    }

# TEST
result = predict_from_journal(
    age=22,
    gender="Male",
    journal_text="I feel overwhelmed with deadlines and not sleeping well. I feel lonely and stressed."
)

print(result)


{'mental_state': 'Healthy', 'confidence_score': np.float64(87.44), 'features': {'stress': 2, 'anxiety': 0, 'depression': 1, 'sleep_quality': 9, 'social_support': 0, 'physical_activity': 0, 'work_pressure': 0}}


In [ ]:
#Api with Flask

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class JournalInput(BaseModel):
    age: int
    gender: str
    journal: str

@app.post("/predict")
def predict(input: JournalInput):
    result = predict_from_journal(
        input.age,
        input.gender,
        input.journal
    )
    return result
